In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('../data/diabetic_data.csv')
df.replace('?', np.nan, inplace=True)


In [4]:
print(f"Starting shape: {df.shape}")


Starting shape: (101766, 50)


In [5]:
print(f"Total encounters: {len(df)}")
print(f"Unique patients: {df['patient_nbr'].nunique()}")
print(f"Duplicate patients: {len(df) - df['patient_nbr'].nunique()}")



Total encounters: 101766
Unique patients: 71518
Duplicate patients: 30248


In [6]:
df = df.drop_duplicates(subset='patient_nbr', keep='first')
print(f"Shape after deduplication: {df.shape}")

Shape after deduplication: (71518, 50)


In [7]:
# Remove patients who died or went to hospice — they can't be readmitted
# This isn't a real "not readmitted" outcome, it's data leakage
dead_hospice_codes = [11, 13, 14, 19, 20, 21]

print(f"Rows before removing deceased/hospice: {len(df)}")
df = df[~df['discharge_disposition_id'].isin(dead_hospice_codes)]
print(f"Rows after: {len(df)}")

Rows before removing deceased/hospice: 71518
Rows after: 69973


In [8]:
columns_to_drop = [
    'encounter_id',      # Just an ID number
    'patient_nbr',       # Patient ID — already used for deduplication
    'weight',            # 97% missing
    'payer_code',        # Insurance type — not clinically relevant
    'examide',           # Only one unique value
    'citoglipton',       # Only one unique value
]

df.drop(columns=columns_to_drop, inplace=True)
print(f"Shape after dropping columns: {df.shape}")

Shape after dropping columns: (69973, 44)


In [9]:
df['race'] = df['race'].fillna('Unknown')
df['medical_specialty'] = df['medical_specialty'].fillna('Unknown')
df['A1Cresult'] = df['A1Cresult'].fillna('No_test')
df['max_glu_serum'] = df['max_glu_serum'].fillna('No_test')
df['diag_1'] = df['diag_1'].fillna('Unknown')
df['diag_2'] = df['diag_2'].fillna('Unknown')
df['diag_3'] = df['diag_3'].fillna('Unknown')

print(f"Total missing values: {df.isnull().sum().sum()}")

Total missing values: 0


In [10]:
print(f"Unique specialties before: {df['medical_specialty'].nunique()}")

specialty_counts = df['medical_specialty'].value_counts()
rare_specialties = specialty_counts[specialty_counts < 50].index
df['medical_specialty'] = df['medical_specialty'].apply(
    lambda x: 'Other' if x in rare_specialties else x
)

print(f"Unique specialties after: {df['medical_specialty'].nunique()}")
print(df['medical_specialty'].value_counts())

Unique specialties before: 71
Unique specialties after: 33
medical_specialty
Unknown                              33639
InternalMedicine                     10641
Family/GeneralPractice                4978
Emergency/Trauma                      4393
Cardiology                            4207
Surgery-General                       2205
Orthopedics                           1128
Orthopedics-Reconstructive            1041
Radiologist                            821
Nephrology                             797
Pulmonology                            637
Psychiatry                             613
ObstetricsandGynecology                593
Urology                                530
Surgery-Cardiovascular/Thoracic        488
Other                                  473
Surgery-Neuro                          404
Gastroenterology                       383
Surgery-Vascular                       359
Oncology                               205
Pediatrics                             195
PhysicalMedicineandR

In [11]:
def map_diagnosis(code):
    if pd.isnull(code) or code == 'Unknown':
        return 'Unknown'
    
    code = str(code)
    
    if code.startswith('E'):
        return 'External'
    if code.startswith('V'):
        return 'Supplementary'
    
    try:
        numeric_code = float(code)
    except ValueError:
        return 'Unknown'
    
    if 250 <= numeric_code < 251:
        return 'Diabetes'
    elif 390 <= numeric_code < 460 or numeric_code == 785:
        return 'Circulatory'
    elif 460 <= numeric_code < 520 or numeric_code == 786:
        return 'Respiratory'
    elif 520 <= numeric_code < 580 or numeric_code == 787:
        return 'Digestive'
    elif 580 <= numeric_code < 630:
        return 'Genitourinary'
    elif 710 <= numeric_code < 740:
        return 'Musculoskeletal'
    elif 140 <= numeric_code < 240:
        return 'Neoplasms'
    elif 240 <= numeric_code < 280:
        return 'Endocrine_Other'
    elif 280 <= numeric_code < 290:
        return 'Blood'
    elif 290 <= numeric_code < 320:
        return 'Mental'
    elif 320 <= numeric_code < 390:
        return 'Nervous'
    elif 680 <= numeric_code < 710:
        return 'Skin'
    elif 780 <= numeric_code < 800:
        return 'Symptoms'
    elif 800 <= numeric_code < 1000:
        return 'Injury'
    else:
        return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col + '_grouped'] = df[col].apply(map_diagnosis)
    print(f"\n{col} categories:")
    print(df[col + '_grouped'].value_counts())

df.drop(columns=['diag_1', 'diag_2', 'diag_3'], inplace=True)
print(f"\nShape: {df.shape}")


diag_1 categories:
diag_1_grouped
Circulatory        21384
Respiratory         9486
Digestive           6487
Diabetes            5748
Injury              4694
Musculoskeletal     4064
Genitourinary       3414
Neoplasms           2538
Other               2312
Symptoms            2233
Endocrine_Other     1850
Skin                1780
Mental              1545
Supplementary        918
Nervous              858
Blood                651
Unknown               10
External               1
Name: count, dtype: int64

diag_2 categories:
diag_2_grouped
Circulatory        22079
Diabetes            9700
Respiratory         6925
Endocrine_Other     5605
Genitourinary       5042
Digestive           2854
Symptoms            2241
Skin                2228
Blood               2075
Mental              1856
Injury              1824
Other               1676
Neoplasms           1599
Musculoskeletal     1295
Supplementary       1217
Nervous              894
External             570
Unknown              293
Name

In [12]:
df['target'] = (df['readmitted'] == '<30').astype(int)
df.drop(columns=['readmitted'], inplace=True)

print(f"Target distribution:")
print(df['target'].value_counts())
print(f"\nPositive class rate: {df['target'].mean()*100:.1f}%")

Target distribution:
target
0    63696
1     6277
Name: count, dtype: int64

Positive class rate: 9.0%


In [13]:
age_mapping = {
    '[0-10)': 0,
    '[10-20)': 1,
    '[20-30)': 2,
    '[30-40)': 3,
    '[40-50)': 4,
    '[50-60)': 5,
    '[60-70)': 6,
    '[70-80)': 7,
    '[80-90)': 8,
    '[90-100)': 9
}

df['age_numeric'] = df['age'].map(age_mapping)
df.drop(columns=['age'], inplace=True)

print(df['age_numeric'].value_counts().sort_index())

age_numeric
0      153
1      534
2     1121
3     2692
4     6828
5    12349
6    15684
7    17750
8    11102
9     1760
Name: count, dtype: int64


In [14]:
med_columns = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
               'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
               'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
               'miglitol', 'troglitazone', 'tolazamide', 'insulin',
               'glyburide-metformin', 'glipizide-metformin',
               'glimepiride-pioglitazone', 'metformin-rosiglitazone',
               'metformin-pioglitazone']

med_mapping = {'No': 0, 'Steady': 1, 'Down': 2, 'Up': 3}

for col in med_columns:
    if col in df.columns:
        df[col] = df[col].map(med_mapping)

# Verify one column to make sure it worked
print("Insulin values after encoding:")
print(df['insulin'].value_counts())

Insulin values after encoding:
insulin
0    34258
1    21617
2     7321
3     6777
Name: count, dtype: int64


In [15]:
# max_glu_serum
glu_mapping = {'No_test': 0, 'None': 0, 'Norm': 1, '>200': 2, '>300': 3}
df['max_glu_serum'] = df['max_glu_serum'].map(glu_mapping)

# A1Cresult
a1c_mapping = {'No_test': 0, 'None': 0, 'Norm': 1, '>7': 2, '>8': 3}
df['A1Cresult'] = df['A1Cresult'].map(a1c_mapping)

# change (whether medications were changed)
df['change'] = df['change'].map({'No': 0, 'Ch': 1})

# diabetesMed (whether diabetes medication was prescribed)
df['diabetesMed'] = df['diabetesMed'].map({'No': 0, 'Yes': 1})

# gender
df['gender'] = df['gender'].map({'Male': 0, 'Female': 1, 'Unknown/Invalid': 0})

print("Encoded columns check:")
print(f"max_glu_serum unique: {df['max_glu_serum'].unique()}")
print(f"A1Cresult unique: {df['A1Cresult'].unique()}")
print(f"change unique: {df['change'].unique()}")
print(f"diabetesMed unique: {df['diabetesMed'].unique()}")
print(f"gender unique: {df['gender'].unique()}")

Encoded columns check:
max_glu_serum unique: [0 3 1 2]
A1Cresult unique: [0 2 3 1]
change unique: [0 1]
diabetesMed unique: [0 1]
gender unique: [1 0]


In [16]:
# Save race labels separately BEFORE encoding
race_series = df['race'].copy()
race_series.to_csv('../data/race_labels.csv', index=False)
print(f"Race labels saved: {race_series.value_counts()}")

Race labels saved: race
Caucasian          52292
AfricanAmerican    12625
Unknown             1918
Hispanic            1500
Other               1150
Asian                488
Name: count, dtype: int64


In [17]:
# Check which columns are still text
cat_columns = df.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns to encode: {cat_columns}")
print(f"Shape before encoding: {df.shape}")

df = pd.get_dummies(df, columns=cat_columns, drop_first=True)

print(f"Shape after encoding: {df.shape}")

Categorical columns to encode: ['race', 'medical_specialty', 'diag_1_grouped', 'diag_2_grouped', 'diag_3_grouped']
Shape before encoding: (69973, 44)
Shape after encoding: (69973, 127)


In [18]:
# Make sure no text columns remain
print(f"Any non-numeric columns: {df.select_dtypes(include='object').columns.tolist()}")
print(f"Any missing values: {df.isnull().sum().sum()}")
print(f"Final shape: {df.shape}")

# Save preprocessed data
df.to_csv('../data/preprocessed_data.csv', index=False)
print("Preprocessed data saved.")

Any non-numeric columns: []
Any missing values: 0
Final shape: (69973, 127)
Preprocessed data saved.
